# Experiment 22 — Theme Needs + Description + Stage — Batch Candidate Relevance Validation

No record key or artificial item ID is sent to the model.

## What the LLM sees

Theme Business Needs + Theme Description + unique Stage data + Stage candidate L3s.

**Not sent:** record key, ground truth, L1/L2 hierarchy.

## Configuration and data

In [ ]:
from pathlib import Path
from time import perf_counter
import ast, json, os
import pandas as pd
from IPython.display import display
from common import call_llm_with_metrics, load_gateway, parse_json_response, save_results_excel, score_sets

HERE=Path.cwd()
def resolve(name, env):
    if os.getenv(env): return Path(os.environ[env]).expanduser()
    for root in (HERE,HERE.parent,HERE/'l3_experiments'):
        p=root/name
        if p.exists(): return p
    raise FileNotFoundError(name)
DATASET_PATH=resolve('golden_valid_set.parquet','L3_GOLDEN_VALID_SET_PATH')
GT_PATH=resolve('results/epic_l3_ground_truth_full_golden.xlsx','L3_GROUND_TRUTH_PATH')
SAMPLE_SIZE=100
SAMPLE_SEED=42

EXPERIMENT_NAME='E22_THEME_NEEDS_DESCRIPTION_STAGE_BATCH_CANDIDATE_RELEVANCE'


## Load combined valid set and Jira GT

In [ ]:
def clean(v):
    if v is None: return ''
    try:
        if pd.isna(v): return ''
    except (TypeError,ValueError): pass
    return str(v).strip()

def parsed(v):
    if v is None: return None
    if hasattr(v,'tolist') and not isinstance(v,(str,bytes)): v=v.tolist()
    if isinstance(v,(dict,list,tuple,set)): return v
    try:
        if pd.isna(v): return None
    except (TypeError,ValueError): pass
    s=str(v).strip()
    if not s: return None
    for fn in (json.loads,ast.literal_eval):
        try: return fn(s)
        except Exception: pass
    return s

def as_list(v):
    v=parsed(v)
    if v is None: return []
    if isinstance(v,(list,tuple,set)): return [clean(x) for x in v if clean(x)]
    return [clean(v)] if clean(v) else []

def col(df,*names,required=True):
    lower={str(c).lower():c for c in df.columns}
    for n in names:
        if n in df.columns: return n
        if n.lower() in lower: return lower[n.lower()]
    if required: raise KeyError(f'Missing column; expected one of {names}')

def gt_map():
    g=pd.read_excel(GT_PATH,sheet_name='jira_l3_ground_truth',dtype=str)
    k,l=col(g,'epic_key'),col(g,'l3_capability_id')
    s=col(g,'status',required=False)
    if s: g=g[g[s].fillna('').str.lower().eq('ok')]
    g=g[g[l].notna()].copy(); g[l]=g[l].map(clean)
    return {clean(k0):sorted(set(x[l])) for k0,x in g.groupby(k,sort=False)}

def load_population():
    f=pd.read_parquet(DATASET_PATH)
    req={
      'theme_key':col(f,'theme_key','key'),'record_key':col(f,'epic_key'),
      'stage_ids':col(f,'stage_ids'),'candidate_l3_ids':col(f,'candidate_l3_ids'),
      'theme_description':col(f,'theme_description','description'),
      'theme_business_needs':col(f,'theme_business_needs','businessNeeds')}
    opt={
      'stage_data':col(f,'stage_data','value_stream_stages','stage_contexts',required=False),
      'candidate_l3_capabilities':col(f,'candidate_l3_capabilities','candidate_l3s',required=False),
      'candidate_l3_by_stage':col(f,'candidate_l3_by_stage','stage_candidate_l3_ids','stage_candidate_l3_capabilities',required=False)}
    p=pd.DataFrame({k:f[v] for k,v in req.items()})
    for k,v in opt.items(): p[k]=f[v] if v else None
    for k in ('theme_key','record_key','theme_description','theme_business_needs'): p[k]=p[k].map(clean)
    p=p.drop_duplicates(['theme_key','record_key']).sort_values(['theme_key','record_key']).reset_index(drop=True)
    if len(p)<SAMPLE_SIZE: raise ValueError(f'Need {SAMPLE_SIZE} rows, found {len(p)}')
    p=p.sample(SAMPLE_SIZE,random_state=SAMPLE_SEED,replace=False)
    gm=gt_map(); p['gt_l3_ids']=p['record_key'].map(gm)
    miss=p[p.gt_l3_ids.isna()].record_key.tolist()
    if miss: raise ValueError(f'GT workbook missing sampled records: {miss}')
    return p.sort_values(['theme_key','record_key']).reset_index(drop=True)

evaluation_population=load_population()
print(f'Selected {len(evaluation_population)} rows from golden_valid_set.parquet with seed={SAMPLE_SEED} across {evaluation_population.theme_key.nunique()} Themes.')
display(evaluation_population.head(100))


## Build Stage payloads

In [ ]:
STAGE_PATH = resolve("VSSrv.csv", "L3_STAGE_PATH")
STAGE_CAPABILITY_MAP_PATH = resolve(
    "VSSCaprv (1).csv",
    "L3_STAGE_CAPABILITY_MAP_PATH",
)

stage_frame = pd.read_csv(
    STAGE_PATH,
    dtype=str,
    encoding="cp1252",
    encoding_errors="replace",
)
stage_capability_map = pd.read_csv(
    STAGE_CAPABILITY_MAP_PATH,
    dtype=str,
    encoding="cp1252",
    encoding_errors="replace",
)


def stage_context(stage_id):
    """Return the full governed metadata for one Value Stream Stage."""
    stage_id = clean(stage_id)
    match = stage_frame.loc[
        stage_frame["Value Stream Stage ID"].astype(str).str.strip().eq(stage_id)
    ]

    if match.empty:
        raise KeyError(f"No Stage metadata found for {stage_id}")

    row = match.iloc[0]
    return {
        "stage_id": stage_id,
        "stage_name": clean(row["Value Stream Stage Name"]),
        "stage_description": clean(row["Value Stream Stage Description"]),
        "entrance_criteria": clean(row["Value Stream Stage Entrance Criteria"]),
        "exit_criteria": clean(row["Value Stream Stage Exit Criteria"]),
    }


def candidate_rows_for_stage(stage_id, allowed_candidate_ids):
    """Return full governed L3 metadata for candidates belonging to one Stage."""
    stage_id = clean(stage_id)
    rows = stage_capability_map.loc[
        stage_capability_map["Value Stream Stage ID"]
        .astype(str)
        .str.strip()
        .eq(stage_id)
    ].copy()

    allowed = {clean(value) for value in allowed_candidate_ids if clean(value)}
    if allowed:
        rows = rows.loc[
            rows["Capability ID"].astype(str).str.strip().isin(allowed)
        ]

    rows = (
        rows
        .drop_duplicates(subset=["Capability ID"], keep="first")
        .sort_values(["Capability Name", "Capability ID"], kind="stable")
    )

    return [
        {
            "capability_id": clean(row["Capability ID"]),
            "capability_name": clean(row["Capability Name"]),
            "capability_description": clean(row["Capability Description"]),
            "capability_tier": clean(row["Capability Tier"]),
        }
        for _, row in rows.iterrows()
    ]


def row_stages(row):
    """Build Stage-specific prompt payloads for one evaluation record."""
    stage_ids = as_list(row["stage_ids"])
    allowed_candidate_ids = as_list(row["candidate_l3_ids"])

    return [
        {
            "stage": stage_context(stage_id),
            "candidates": candidate_rows_for_stage(
                stage_id,
                allowed_candidate_ids,
            ),
        }
        for stage_id in stage_ids
    ]


def merged_stages(rows):
    """Build one deduplicated Stage payload per Theme batch."""
    stage_to_allowed = {}

    for row in rows.to_dict("records"):
        allowed_candidate_ids = as_list(row["candidate_l3_ids"])
        for stage_id in as_list(row["stage_ids"]):
            stage_to_allowed.setdefault(stage_id, set()).update(
                allowed_candidate_ids
            )

    return [
        {
            **stage_context(stage_id),
            "candidate_l3_capabilities": candidate_rows_for_stage(
                stage_id,
                stage_to_allowed[stage_id],
            ),
        }
        for stage_id in sorted(stage_to_allowed)
    ]


## Production prompt

In [ ]:
SYSTEM_PROMPT = """\
You are performing Level 3 business capability relevance validation for multiple Value Stream Stages that share the same Theme context.

An L3 capability is a Level 3 business capability: a specific business function within the enterprise capability hierarchy.

Use Theme Business Needs as the primary shared business evidence.
Use Theme Description as supporting context that can clarify the Business Needs, but do not use it to introduce unsupported business functions.
Evaluate each supplied Value Stream Stage independently using only the shared Theme context, that Stage's governed metadata, and that Stage's candidate L3 capabilities.

EVIDENCE

Theme Business Needs describes the shared business outcomes, requirements, and functions that need to be delivered.
Theme Description provides supporting scope and intent for those Business Needs.
Each Stage's name, description, entrance criteria, and exit criteria define that Stage's business-process boundary.

For each candidate L3:
- capability_id is the exact identifier of the candidate being judged.
- capability_description is the primary semantic definition of the business function.
- capability_name is a supporting business label.
- capability_tier is supporting taxonomy context only.

Do not infer business meaning from capability_id.

RELEVANCE VALIDATION

For each Stage independently:

1. Identify the business functions expressed by the Theme Business Needs.
2. Use Theme Description to clarify scope and intent.
3. Constrain those functions to this Stage using the Stage name, description, entrance criteria, and exit criteria.
4. Treat each supplied candidate L3 as an independent relevance claim.

For EVERY supplied candidate, decide relevant=true or relevant=false.

Set relevant=true when the candidate's business function is:
- explicitly stated in the Theme context, OR
- strongly semantically implied by the Theme context within this Stage boundary.

The exact capability name or terminology does not need to appear in the Theme text when the underlying business function is clearly supported.

Set relevant=false when the capability is only:
- a member of the supplied Stage,
- a terminology overlap,
- generally related or adjacent,
- a prerequisite,
- upstream or downstream,
- a function that commonly supports another relevant capability without itself being supported by the Theme context.

Judge every candidate independently. A true decision for one candidate neither requires nor excludes a true decision for another candidate.
You MUST evaluate every candidate exactly once. Do not omit any supplied candidate and do not add candidates that were not supplied for that Stage.
Do not use one Stage's metadata or candidates as evidence for another Stage.
Return exactly one result for every supplied stage_id.

OUTPUT

Return JSON only:

{
  "stages": [
    {
      "stage_id": "VSS000123",
      "candidates": [
        {"capability_id": "CAP00000123", "relevant": true},
        {"capability_id": "CAP00000456", "relevant": false}
      ]
    },
    {
      "stage_id": "VSS000456",
      "candidates": []
    }
  ]
}

Do not return reasons, explanations, Markdown, or additional fields.
"""

def _prompt_text(value):
    return "" if value is None else str(value).strip()


def _format_stage_block(stage, candidate_rows, index):
    lines = [
        f"Stage {index}:",
        f"  Stage ID: {_prompt_text(stage.get('stage_id'))}",
        f"  Stage Name: {_prompt_text(stage.get('stage_name'))}",
        f"  Stage Description: {_prompt_text(stage.get('stage_description'))}",
        f"  Entrance Criteria: {_prompt_text(stage.get('entrance_criteria'))}",
        f"  Exit Criteria: {_prompt_text(stage.get('exit_criteria'))}",
        "",
        "  Candidate L3 Capabilities:",
    ]

    if not candidate_rows:
        lines.append("    None")
        return "\n".join(lines)

    for capability_index, capability in enumerate(candidate_rows, start=1):
        lines.extend(
            [
                f"    {capability_index}.",
                f"      Capability ID: {_prompt_text(capability.get('capability_id'))}",
                f"      Capability Name: {_prompt_text(capability.get('capability_name'))}",
                f"      Capability Description: {_prompt_text(capability.get('capability_description'))}",
                f"      Capability Tier: {_prompt_text(capability.get('capability_tier'))}",
            ]
        )

    return "\n".join(lines)


def build_user_prompt(context, stages):
    sections = [
        "Theme Business Needs:",
        _prompt_text(context.get("theme_business_needs")),
    ]

    sections.extend([
        "",
        "Theme Description:",
        _prompt_text(context.get("theme_description")),
    ])

    sections.extend(["", "Value Stream Stages:"])

    for index, stage_payload in enumerate(stages, start=1):
        stage = {
            key: value
            for key, value in stage_payload.items()
            if key != "candidate_l3_capabilities"
        }
        candidate_rows = stage_payload.get("candidate_l3_capabilities", [])
        sections.extend(
            [
                "",
                _format_stage_block(stage, candidate_rows, index),
            ]
        )

    return "\n".join(sections).strip()


def theme_context(row):
    """Return exactly the shared Theme fields that are model-visible."""
    return {
        "theme_business_needs": row["theme_business_needs"],
        "theme_description": row["theme_description"],
    }


## Prompt preview — exactly as sent

In [ ]:
theme,rows=next(iter(evaluation_population.groupby('theme_key',sort=True)))
preview_user_prompt=build_user_prompt(theme_context(rows.iloc[0].to_dict()),merged_stages(rows))
print('SYSTEM PROMPT — EXACT TEXT SENT TO LLM\n'+'='*80+'\n'+SYSTEM_PROMPT+'\n\nUSER PROMPT — EXACT TEXT SENT TO LLM\n'+'='*80+'\n'+preview_user_prompt)

## Prediction and evaluation

In [ ]:
# PROMPT_AUDIT_RESET_E17_E20
from common import reset_prompt_audit_log

reset_prompt_audit_log()


In [ ]:
def validate(payload,expected,allowed):
    if not isinstance(payload,dict) or set(payload)!={'stages'} or not isinstance(payload['stages'],list):
        raise ValueError('Expected stages list')

    expected=set(expected)
    out={}
    decisions=[]
    seen_stages=set()

    for result in payload['stages']:
        if not isinstance(result,dict) or set(result)!={'stage_id','candidates'}:
            raise ValueError('Each result needs stage_id and candidates')

        sid=clean(result['stage_id'])
        if sid not in expected or sid in seen_stages:
            raise ValueError(f'Invalid or duplicate stage_id: {sid}')
        if not isinstance(result['candidates'],list):
            raise ValueError(f'Candidates must be a list for {sid}')

        allowed_ids=set(allowed[sid])
        seen_candidates=set()
        selected=[]

        for candidate in result['candidates']:
            if not isinstance(candidate,dict) or set(candidate) != {'capability_id','relevant'}:
                raise ValueError(f'Each candidate decision for {sid} needs capability_id and relevant')
            if not isinstance(candidate['relevant'], bool):
                raise ValueError(f'relevant must be boolean for {sid}')

            cid=clean(candidate['capability_id'])
            if cid not in allowed_ids or cid in seen_candidates:
                raise ValueError(f'Invalid or duplicate candidate {cid} for {sid}')

            seen_candidates.add(cid)
            relevant=candidate['relevant']
            decisions.append({'stage_id':sid,'capability_id':cid,'relevant':relevant})
            if relevant:
                selected.append(cid)

        if seen_candidates != allowed_ids:
            missing=sorted(allowed_ids-seen_candidates)
            raise ValueError(f'Missing candidate decisions for {sid}: {missing}')

        seen_stages.add(sid)
        out[sid]=selected

    if seen_stages != expected:
        missing=sorted(expected-seen_stages)
        raise ValueError(f'Missing stage results: {missing}')

    return out,decisions


def predict(gateway,ctx,stages):
    u=build_user_prompt(ctx,stages)
    allowed={s['stage_id']:[c['capability_id'] for c in s['candidate_l3_capabilities']] for s in stages}
    raw,m=call_llm_with_metrics(gateway,SYSTEM_PROMPT,u,id='9zdn8n',reasoning_effort='low')
    selected,decisions=validate(parse_json_response(raw),[s['stage_id'] for s in stages],allowed)
    return selected,decisions,m


def run():
    g=load_gateway(); rr=[]; cc=[]; relevance=[]

    for theme,rows in evaluation_population.groupby('theme_key',sort=True):
        rows=rows.reset_index(drop=True)
        stages=merged_stages(rows)
        stage_lookup={s['stage_id']:s for s in stages}
        candidate_lookup={
            (s['stage_id'],c['capability_id']):c
            for s in stages
            for c in s['candidate_l3_capabilities']
        }
        t=perf_counter()

        try:
            pred,decisions,m=predict(g,theme_context(rows.iloc[0].to_dict()),stages)

            for decision in decisions:
                sid=decision['stage_id']; cid=decision['capability_id']
                stage=stage_lookup[sid]
                capability=candidate_lookup[(sid,cid)]
                relevance.append({
                    'experiment':EXPERIMENT_NAME,
                    'theme_key':theme,
                    'stage_id':sid,
                    'stage_name':stage.get('stage_name'),
                    'capability_id':cid,
                    'capability_name':capability.get('capability_name'),
                    'capability_description':capability.get('capability_description'),
                    'capability_tier':capability.get('capability_tier'),
                    'relevant':decision['relevant'],
                })

            cc.append({
                'experiment':EXPERIMENT_NAME,'theme_key':theme,'record_count':len(rows),'stage_count':len(stages),
                'status':'ok','latency_seconds':m.get('latency_seconds'),'input_tokens':m.get('input_tokens'),
                'output_tokens':m.get('output_tokens'),'total_tokens':m.get('total_tokens'),'error':None
            })

            for r in rows.to_dict('records'):
                vals=sorted({cid for sid in as_list(r['stage_ids']) for cid in pred.get(sid,[])})
                truth=r['gt_l3_ids']
                rr.append({
                    'experiment':EXPERIMENT_NAME,'theme_key':theme,'record_key':r['record_key'],
                    'stage_ids':as_list(r['stage_ids']),'predicted_l3_ids':vals,'gt_l3_ids':truth,
                    'status':'ok','error':None,**score_sets(vals,truth)
                })

        except Exception as e:
            err=str(e)
            cc.append({
                'experiment':EXPERIMENT_NAME,'theme_key':theme,'record_count':len(rows),'stage_count':len(stages),
                'status':'error','latency_seconds':perf_counter()-t,'input_tokens':None,'output_tokens':None,
                'total_tokens':None,'error':err
            })
            for r in rows.to_dict('records'):
                truth=r['gt_l3_ids']
                rr.append({
                    'experiment':EXPERIMENT_NAME,'theme_key':theme,'record_key':r['record_key'],
                    'stage_ids':as_list(r['stage_ids']),'predicted_l3_ids':None,'gt_l3_ids':truth,
                    'status':'error','error':err,'exact_match':None,'precision':None,'recall':None,'f1':None,
                    'predicted_count':None,'truth_count':len(truth)
                })

    return pd.DataFrame(rr),pd.DataFrame(cc),pd.DataFrame(relevance)


results,call_metrics,candidate_relevance=run()
scored=results[results.status.eq('ok')]
calls=call_metrics[call_metrics.status.eq('ok')]
summary=pd.DataFrame([{
    'scope':'fixed_100_from_golden_valid_set_seed_42_theme_batch_candidate_relevance',
    'evaluated_records':len(scored),
    'exact_match_accuracy':scored.exact_match.mean() if len(scored) else 0,
    'mean_precision':scored.precision.mean() if len(scored) else 0,
    'mean_recall':scored.recall.mean() if len(scored) else 0,
    'mean_f1':scored.f1.mean() if len(scored) else 0,
}])
latency_tokens=pd.DataFrame([{
    'successful_calls':len(calls),
    'failed_calls':int(call_metrics.status.eq('error').sum()),
    'avg_records_per_call':calls.record_count.mean() if len(calls) else None,
    'avg_stages_per_call':calls.stage_count.mean() if len(calls) else None,
    'avg_latency_seconds':calls.latency_seconds.mean() if len(calls) else None,
    'p50_latency_seconds':calls.latency_seconds.quantile(.5) if len(calls) else None,
    'p95_latency_seconds':calls.latency_seconds.quantile(.95) if len(calls) else None,
    'total_input_tokens':calls.input_tokens.sum() if len(calls) else 0,
    'total_output_tokens':calls.output_tokens.sum() if len(calls) else 0,
    'total_tokens':calls.total_tokens.sum() if len(calls) else 0,
    'tokens_per_scored_record':calls.total_tokens.sum()/len(scored) if len(scored) else None,
}])

display(summary)
display(latency_tokens)
display(call_metrics)
display(candidate_relevance)
display(results.head(100))
print('Saved',save_results_excel(
    results,
    EXPERIMENT_NAME,
    'results',
    extra_sheets={
        'evaluation_summary':summary,
        'llm_metrics':call_metrics,
        'latency_tokens':latency_tokens,
        'evaluation_population':evaluation_population,
        'candidate_relevance':candidate_relevance,
    },
))


## Exact prompt audit from this run

In [ ]:
# PROMPT_AUDIT_EXPORT_E17_E20
from common import get_prompt_audit_log

prompt_log = pd.DataFrame(get_prompt_audit_log())
if not prompt_log.empty:
    prompt_log.insert(0, "experiment", EXPERIMENT_NAME)

prompt_sample = prompt_log.head(1).copy()

print(f"Exact prompt audit entries captured: {len(prompt_log)}")
if not prompt_sample.empty:
    sample = prompt_sample.iloc[0]
    print("\nSYSTEM PROMPT SAMPLE — EXACT TEXT SENT TO LLM")
    print("=" * 80)
    print(sample["system_prompt"])
    print("\nUSER PROMPT SAMPLE — EXACT TEXT SENT TO LLM")
    print("=" * 80)
    print(sample["user_prompt"])

extra_sheets = {
    "call_metrics": call_metrics,
    "prompt_log": prompt_log,
    "prompt_sample": prompt_sample,
    "candidate_relevance": candidate_relevance,
}
if "summary" in globals():
    extra_sheets["run_summary"] = summary

prompt_audit_output_path = save_results_excel(
    results,
    EXPERIMENT_NAME,
    extra_sheets=extra_sheets,
)
print(f"Saved results with prompt audit: {prompt_audit_output_path}")
